# Unstructured fine-tuning training for knowledge-injection

Some notes:
- We have very little data, so the learning rate needs to be fairly high (TODO TEST/VERIFY)
- We have a whole A100 GPU to make use of
    - Don't bother with LoRA, we have more than enough VRAM to fine-tune the entire model, this should help compensate for a low amount of data somewhat
    - Don't bother with quantization, see above: more than enough VRAM, we don't want to sacrifice training results

In [ ]:
# Hyperparameters
MAX_LENGTH = 2048
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-6
NUM_EPOCHS = 10
WARMUP_STEPS = 200
SAVE_STEPS = 500

# Base model we will use
MODEL = "Qwen/Qwen2.5-14B-Instruct"
# Fine-tuned Model
OUT_MODEL_DIR = f"./{MODEL}-Finetuned"
# Folder containing source documents to fine-tune on
DOC_DIR = "/opt/shared/data/raw"

In [ ]:
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    try:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    except Exception:
        # best-effort: some environments don't expose a device name
        print("GPU: available (name unavailable)")
else:
    print("GPU: None")


In [ ]:
# load the data

print("Loading documents...")

documents = []
for entry in os.scandir(DOC_DIR):  
    if entry.is_file():
        with open(entry.path, "r", encoding="utf-8") as f:
            documents.append(f.read())


print("Loading documents done.")

In [ ]:
# Load tokenizer

print(f"Loading tokenizer: {MODEL}")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
# Ensure padding token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading tokenizer: {MODEL} done")

chunks = []

for doc in documents:
    # Split by double newlines (paragraph boundaries)
    paragraphs = [p.strip() for p in doc.split("\n\n") if p.strip()]
    
    current_chunk = []
    current_length = 0
    
    for para in paragraphs:
        para_tokens = tokenizer(para, truncation=False)["input_ids"]
        para_length = len(para_tokens)

        # If single paragraph exceeds max_length, add both the current chunk and itself as a chunk
        if para_length > MAX_LENGTH:
            # Save current chunk if it exists
            if current_chunk:
                chunk_text = "\n\n".join(current_chunk)
                chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                chunks.append({"input_ids": chunk_ids})
                current_chunk = []
                current_length = 0
            
            # Split long paragraph into sentences
            sentences = para.split(". ")
            temp_chunk = []
            temp_length = 0
            
            for sent in sentences:
                sent_tokens = tokenizer(sent, truncation=False)["input_ids"]
                if temp_length + len(sent_tokens) <= max_length:
                    temp_chunk.append(sent)
                    temp_length += len(sent_tokens)
                else:
                    if temp_chunk:
                        chunk_text = ". ".join(temp_chunk)
                        chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                        chunks.append({"input_ids": chunk_ids})
                    temp_chunk = [sent]
                    temp_length = len(sent_tokens)
            
            if temp_chunk:
                chunk_text = ". ".join(temp_chunk)
                chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                chunks.append({"input_ids": chunk_ids})
            
            continue
        
        # If adding this paragraph would exceed MAX_LENGTH, save current chunk
        if current_length + para_length > MAX_LENGTH:
            if current_chunk:
                chunk_text = "\n\n".join(current_chunk)
                chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
                chunks.append({"input_ids": chunk_ids})
            current_chunk = [para]
            current_length = para_length
        else:
            current_chunk.append(para)
            current_length += para_length
    
    # Save remaining chunk
    if current_chunk:
        chunk_text = "\n\n".join(current_chunk)
        chunk_ids = tokenizer(chunk_text, truncation=True, max_length=MAX_LENGTH)["input_ids"]
        chunks.append({"input_ids": chunk_ids})

print(f"Created {len(chunks)} chunks (avg length: {sum(len(c['input_ids']) for c in chunks) / len(chunks):.0f} tokens)")

dataset = Dataset.from_list(chunks)

# split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
# train_dataset = split_dataset["train"]
# eval_dataset = split_dataset["test"]
train_dataset = dataset
eval_dataset = None

# Data collator for causal LM (handles labels automatically)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # causal LM, not masked LM
)

In [ ]:
# Load the model

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

# The cache is not useful during fine-tuning training, it's a KV cache aimed at inference
model.config.use_cache = False

print(f"Model dtype: {model.dtype}")
print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# Perform the continues pre-training / unstructured fine-tuning

training_args = TrainingArguments(
    output_dir=OUT_MODEL_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=4,
    logging_first_step=True,
    save_steps=SAVE_STEPS,
    eval_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,  # saves memory, turning this off should be 10-15% faster but also more risky 
    # optim="adamw_torch",
    optim="adafactor",            # essential, adamw stores 2 copies of weights and will run out of memory - adafactor is more memory-efficient
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

print(f"Saving finetuned model to {OUT_MODEL_DIR}")
trainer.save_model(OUT_MODEL_DIR)
tokenizer.save_pretrained(OUT_MODEL_DIR)

print("Training complete.")

In [ ]:
# while True:
#     print("> ", end="")
#     userinput = input()

#     if not userinput:
#         continue

#     if userinput == "quit" or userinput == "q" or userinput == "exit":
#         break

#     pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=300)
#     result = pipe(f"<s>[INST] {userinput} [/INST]")
#     print(result[0]['generated_text'])

